
A 2x2 grid of 2x2 systolic blocks acting as one logical 4x4 array.

The blocks are not configured. Each one learns its position from the first
header it receives, so the host only sends data and starts all four.

## Node

In [2]:
import numpy as np
import hyperfpga_cluster as hfc
from hyperfpga_cluster import configure_logging

configure_logging()
test_nodes = await hfc.request_nodes('4ge21')
print(test_nodes[0]['fpga'])

FIRMWARE = "sa_q4_ILA2" 
IP_NAME  = "sa_grid"

{'model': '4ge21', 'state': 'unknown', 'firmware': ''}


## Constants

In [3]:
PE, R, S, K = 2, 2, 2, 4
N, NC = R*PE, S*PE
W     = PE * 16                  # bus width in bits
APB   = W // 32                  # accumulators per beat
CBEAT = (PE*PE) // APB           # beats of one block's C tile

MAGIC = 0x53
A_FRAME, B_FRAME = 0, 1

# Two separate DMA transfers. The DMA puts TLAST at the end of a transfer, and
# load reads a frame group until TLAST, so the split is what tells a block
# where the B group ends and the A group begins.
B_BEATS    = S * (1+K)
A_BEATS    = R * (1+K)
RECV_BEATS = 1 + R*S*CBEAT       # one header, then the gathered C

# Tile shape is fixed by the bitstream. One accelerator call computes exactly
# a TM x TP tile of C from a TM x TK tile of A and a TK x TP tile of B.
TM, TP, TK = N, NC, K

# ---- the only numbers to change ----------------------------------------
M_FULL, P_FULL, K_FULL = 4, 4, 4
# ------------------------------------------------------------------------

print(f"send {B_BEATS} + {A_BEATS} beats per tile, receive {RECV_BEATS}")
print(f"tile shape {TM}x{TP}x{TK}")


send 10 + 10 beats per tile, receive 17
tile shape 4x4x4


## Build the stream

One header then K beats per frame. Each beat packs PE operands, low half first.
`dst_r` and `dst_s` name the block that should keep the frame; that is how a
block discovers where it is.

In [4]:
def hdr(type_, dst_r, dst_s, ln):
    return (MAGIC << 24) | (type_ << 20) | (dst_r << 16) | (dst_s << 12) | ln

def pack(vals):
    w = 0
    for n, v in enumerate(vals):
        w |= (int(v) & 0xFFFF) << (16*n)
    return w

def build_stream(A, B):
    s = []
    for col in range(S):                       # transfer 1: B frames
        s.append(hdr(B_FRAME, 0, col, K))
        for k in range(K):
            s.append(pack(B[k, col*PE:(col+1)*PE]))
    for row in range(R):                       # transfer 2: A frames
        s.append(hdr(A_FRAME, row, 0, K))
        for k in range(K):
            s.append(pack(A[row*PE:(row+1)*PE, k]))
    return s

def unpack_C(beats):
    vals = []
    for w in beats[1:]:                        # drop the header
        for m in range(APB):
            v = (int(w) >> (32*m)) & 0xFFFFFFFF
            vals.append(v - (1 << 32) if v >> 31 else v)
    C, p = np.zeros((N, NC), dtype=np.int64), 0
    for r in range(R):
        for s_ in range(S):
            for i in range(PE):
                for j in range(PE):
                    C[r*PE+i][s_*PE+j] = vals[p]; p += 1
    return C

In [5]:
def tile_streams(A, B):
    """Split A @ B into tile products the accelerator can run.

    Returns the per-tile streams, the (row, col) each result belongs to, and
    the tile counts. A and B are zero padded up to a whole number of tiles, so
    M_FULL, P_FULL and K_FULL do not have to be multiples of the tile shape.
    """
    m, kf = A.shape
    kf2, p = B.shape
    assert kf == kf2

    mt = -(-m // TM)
    pt = -(-p // TP)
    kt = -(-kf // TK)

    Ap = np.zeros((mt*TM, kt*TK), dtype=np.int64)
    Bp = np.zeros((kt*TK, pt*TP), dtype=np.int64)
    Ap[:m, :kf] = A
    Bp[:kf, :p] = B

    streams, index = [], []
    for i in range(mt):
        for j in range(pt):
            for k in range(kt):
                streams.append(build_stream(Ap[i*TM:(i+1)*TM, k*TK:(k+1)*TK],
                                            Bp[k*TK:(k+1)*TK, j*TP:(j+1)*TP]))
                index.append((i, j))
    return streams, index, (mt, pt, kt)


def assemble_C(all_beats, index, shape):
    """Accumulate the tile results into the full C matrix."""
    m, p = shape
    mt = -(-m // TM)
    pt = -(-p // TP)
    C = np.zeros((mt*TM, pt*TP), dtype=np.int64)
    for beats, (i, j) in zip(all_beats, index):
        C[i*TM:(i+1)*TM, j*TP:(j+1)*TP] += unpack_C(beats)
    return C[:m, :p]


## Test data

Bounded so the 32-bit accumulator cannot overflow.

In [6]:
AMAX = 1000
assert TK * AMAX * AMAX < 2**31          # per tile, the accumulator is 32 bit

rng = np.random.default_rng(0)
A = rng.integers(-AMAX, AMAX+1, size=(M_FULL, K_FULL)).astype(np.int64)
B = rng.integers(-AMAX, AMAX+1, size=(K_FULL, P_FULL)).astype(np.int64)
C_golden = A @ B

streams, index, (mt, pt, kt) = tile_streams(A, B)
n_tiles = len(streams)
in_beats = B_BEATS + A_BEATS

print(f"{M_FULL}x{K_FULL} @ {K_FULL}x{P_FULL}")
print(f"{mt} x {pt} x {kt} tiles = {n_tiles} accelerator calls")
print(f"buffer needed = {n_tiles*(in_beats+RECV_BEATS)*(W//8)} bytes")


4x4 @ 4x4
1 x 1 x 1 tiles = 1 accelerator calls
buffer needed = 148 bytes


## Run

In [7]:
def run_tiles(streams, ip_name, n_blocks, b_beats, a_beats,
              recv_beats, wbytes, runs=1, chunk=None):
    """Run every tile through the accelerator inside one remote call.

    Tiles are processed in chunks that fit the DMA buffer, so the problem size
    is limited by time rather than by buffer capacity. Writing a chunk into the
    buffer and reading its results back both happen outside the timer, so what
    is measured is the DMA loop alone. The whole pass is repeated `runs` times
    and every duration is returned, so the caller can drop warm up runs and
    take a median.
    """
    import os, time
    import hyperfpga_comutils as hf

    base, inv = "/sys/class/uio", []
    for dev in sorted(os.listdir(base)):
        try:
            nm = open(f"{base}/{dev}/name").read().strip()
            ad = int(open(f"{base}/{dev}/maps/map0/addr").read().strip(), 16)
        except Exception:
            continue
        inv.append((dev, nm, ad))

    dma  = list(hf.HardwareManager.detect()["dmas"].values())[0]
    devs = [d for d, n, a in sorted(inv, key=lambda x: x[2]) if n == ip_name]
    if len(devs) != n_blocks:
        raise RuntimeError(f"expected {n_blocks} '{ip_name}' uio nodes, found {devs}")
    blk = [hf.HlsControl(f"/dev/{d}") for d in devs]

    buf = hf.UDMABuffer(0, format_in="unsigned32", format_out="unsigned32",
                        signed=False)

    nt       = len(streams)
    in_beats = b_beats + a_beats
    per_tile = (in_beats + recv_beats) * wbytes

    fit = max(1, int(buf.size * 0.9) // per_tile)
    chunk = min(chunk or fit, fit, nt)

    OFF_IN = 0
    OFF_C  = chunk * in_beats * wbytes

    totals, beats = [], None
    for _ in range(runs):
        elapsed = 0.0
        collect = [] if beats is None else None

        for c0 in range(0, nt, chunk):
            cs = streams[c0:c0 + chunk]
            nc = len(cs)

            flat = [int(x) for s in cs for x in s]
            buf.write_values(flat, OFF_IN)
            buf.sync_for_device(OFF_IN, nc * in_beats * wbytes)

            t0 = time.perf_counter()
            for n in range(nc):
                p_in  = buf.phys_addr + OFF_IN + n * in_beats * wbytes
                p_out = buf.phys_addr + OFF_C  + n * recv_beats * wbytes

                dma.start_s2mm(p_out, recv_beats * wbytes)
                for b in blk:
                    b.start()
                dma.start_mm2s(p_in, b_beats * wbytes)
                dma.wait_mm2s(5.0)
                dma.start_mm2s(p_in + b_beats * wbytes, a_beats * wbytes)
                dma.wait_mm2s(5.0)
                dma.wait_s2mm(5.0)
            t1 = time.perf_counter()
            elapsed += t1 - t0

            if collect is not None:
                buf.sync_for_cpu(OFF_C, nc * recv_beats * wbytes)
                fo = [int(x) for x in buf.read_values(nc * recv_beats, OFF_C)]
                collect += [fo[i*recv_beats:(i+1)*recv_beats] for i in range(nc)]

        totals.append(elapsed)
        if collect is not None:
            beats = collect

    return {"dma": totals, "tiles": nt, "chunk": chunk,
            "buf_size": buf.size, "beats": beats, "blocks": devs}


In [8]:
PROGRAM = True         # set False once the firmware is already loaded
RUNS    = 12           # WARMUP runs are discarded, the rest give the median
WARMUP  = 2

cluster = (hfc.HyperFPGACluster(nodes=test_nodes, firmware=FIRMWARE)
           if PROGRAM else hfc.HyperFPGACluster(nodes=test_nodes))
rc, t, C_hw = None, None, None
try:
    await cluster.configure()
    cluster.create_profile(mpi=False)
    rc = await cluster.start_and_connect()
    rc.wait_for_engines(n=1, timeout=60)
    t = rc[0].apply_sync(run_tiles, streams, IP_NAME, R * S, B_BEATS, A_BEATS,
                         RECV_BEATS, W // 8, RUNS)
    C_hw = assemble_C(t["beats"], index, (M_FULL, P_FULL))
    print("blocks:", t["blocks"], " tiles:", t["tiles"], " runs:", len(t["dma"]))
    print(f"buffer {t['buf_size']} bytes, {t['chunk']} tiles per chunk")
finally:
    pass


Starting 1 engines with <class 'ipyparallel.cluster.launcher.SSHEngineSetLauncher'>
ensuring remote mlabadm@192.168.0.4:.ipython/profile_ssh/security/ exists
sending /home/salmutairi/.ipython/profile_ssh/security/ipcontroller-1787084275-86qw-client.json to mlabadm@192.168.0.4:.ipython/profile_ssh/security/ipcontroller-1787084275-86qw-client.json
ensuring remote mlabadm@192.168.0.4:.ipython/profile_ssh/security/ exists
sending /home/salmutairi/.ipython/profile_ssh/security/ipcontroller-1787084275-86qw-engine.json to mlabadm@192.168.0.4:.ipython/profile_ssh/security/ipcontroller-1787084275-86qw-engine.json
Running `/opt/hfpga-ml311/bin/ipengine --profile-dir=/home/mlabadm/.ipython/profile_ssh`


  0%|          | 0/1 [00:00<?, ?engine/s]

blocks: ['uio5', 'uio6', 'uio7', 'uio8']  tiles: 1  runs: 12
buffer 4194304 bytes, 1 tiles per chunk


## 6. Check

In [58]:
h = t["beats"][0][0]
print(f"header magic 0x{(h >> 24) & 0xFF:02X}  (expect 0x{MAGIC:02X})")

err = np.abs(C_hw - C_golden)
print(f"max error = {err.max()}")
print("PASS" if err.max() == 0 else "FAIL")


header magic 0x53  (expect 0x53)
max error = 0
PASS


In [59]:
d    = np.array(t["dma"][WARMUP:]) * 1e6      # us, warm up discarded
nt   = t["tiles"]
ops  = 2 * M_FULL * P_FULL * K_FULL
FCLK = 299.997e6
med  = np.median(d)

print(f"problem        {M_FULL}x{K_FULL} @ {K_FULL}x{P_FULL}   ({nt} tiles)")
print(f"runs used      {len(d)} of {len(t['dma'])}")
print()
print(f"DMA loop       median {med:9.2f} us   min {d.min():8.2f}   max {d.max():8.2f}")
print(f"spread         {(d.max()-d.min())/med*100:9.1f} %")
print(f"per tile       {med/nt:9.3f} us  = {med/nt*1e-6*FCLK:8.0f} cycles")
print(f"throughput     {ops/(med*1e-6)/1e9:9.4f} GOPS")
print()
array_us = nt * 75 / FCLK * 1e6               # 75 cycles per tile, from the ILA
print(f"array only     {array_us:9.3f} us   ({array_us/med*100:.2f}% of the loop)")
print(f"overhead       {med-array_us:9.3f} us   ({(med-array_us)/med*100:.2f}%)")


problem        4x4 @ 4x4   (1 tiles)
runs used      10 of 12

DMA loop       median     18.66 us   min    18.39   max    19.11
spread               3.9 %
per tile          18.660 us  =     5598 cycles
throughput        0.0069 GOPS

array only         0.250 us   (1.34% of the loop)
overhead          18.410 us   (98.66%)


In [60]:
# Does the per tile cost stay flat as the problem grows? If it does, the
# descriptor turnaround is paid once per tile and never amortises, and the
# only fix is to batch tiles into fewer DMA transfers.
#
# Cost grows as size**3. At 128 the run is 32768 tiles, so the number of
# repeats is reduced for the larger sizes to keep the sweep to a few minutes.

SWEEP_SIZES = [4, 8, 16, 32, 64, 128]

def runs_for(n_tiles):
    if n_tiles <= 512:
        return 5
    if n_tiles <= 4096:
        return 3
    return 2

rows = []
for size in SWEEP_SIZES:
    Aa = rng.integers(-AMAX, AMAX+1, size=(size, size)).astype(np.int64)
    Bb = rng.integers(-AMAX, AMAX+1, size=(size, size)).astype(np.int64)
    st, ix, _ = tile_streams(Aa, Bb)
    nr = runs_for(len(st))
    print(f"size {size:4d}  {len(st):6d} tiles  x{nr} ...", flush=True)

    tt  = rc[0].apply_sync(run_tiles, st, IP_NAME, R * S, B_BEATS, A_BEATS,
                           RECV_BEATS, W // 8, nr)
    Ch  = assemble_C(tt["beats"], ix, (size, size))
    ok  = np.array_equal(Ch, Aa @ Bb)
    dd  = np.array(tt["dma"][1:]) if nr > 1 else np.array(tt["dma"])
    med = float(np.median(dd))
    rows.append((size, tt["tiles"], med, med/tt["tiles"],
                 2*size**3/med/1e9, ok))

print()
print(f"{'size':>5} {'tiles':>7} {'total ms':>11} {'per tile us':>12} "
      f"{'GOPS':>9}  ok")
for size, ntl, tot, per, gops, ok in rows:
    print(f"{size:5d} {ntl:7d} {tot*1e3:11.2f} {per*1e6:12.3f} {gops:9.4f}  {ok}")

if len(rows) > 1:
    r = rows[-1][3] / rows[0][3]
    print(f"\nper tile cost changed by {r:.2f}x from {rows[0][0]} to {rows[-1][0]}")
    print("close to 1.00 means the descriptor turnaround never amortises")


size    4       1 tiles  x5 ...
size    8       8 tiles  x5 ...
size   16      64 tiles  x5 ...
size   32     512 tiles  x5 ...
size   64    4096 tiles  x3 ...
size  128   32768 tiles  x2 ...

 size   tiles    total ms  per tile us      GOPS  ok
    4       1        0.02       19.590    0.0065  True
    8       8        0.11       14.029    0.0091  True
   16      64        0.89       13.971    0.0092  True
   32     512        8.16       15.937    0.0080  True
   64    4096       59.43       14.509    0.0088  True
  128   32768      477.11       14.560    0.0088  True

per tile cost changed by 0.74x from 4 to 128
close to 1.00 means the descriptor turnaround never amortises


In [61]:
# Per tile latency of the whole grid, in cycles.
CYC_GRID       = 75            # logic analyzer, operand in -> last result out
CYC_LO, CYC_HI = 67, 156       # cosimulation bound for the grid
FCLK           = 299.997e6

assert CYC_LO <= CYC_GRID <= CYC_HI, "measured grid latency is outside the bound"

hdr = (f"{'size':>5} {'tiles':>7} {'T_total ms':>11} {'T_comp ms':>10} "
       f"{'T_over ms':>10} {'comp %':>7} {'GOPS sys':>9} {'GOPS arr':>9}")
print(hdr)
print("-" * len(hdr))

table = []
for size, ntl, tot, per, gops, ok in rows:
    t_comp = ntl * CYC_GRID / FCLK
    t_over = tot - t_comp
    ops    = 2 * size**3
    row = dict(size=size, tiles=ntl, ops=ops,
               t_total_s=tot, t_compute_s=t_comp, t_over_s=t_over,
               compute_pct=100*t_comp/tot,
               t_compute_lo_s=ntl*CYC_LO/FCLK,
               t_compute_hi_s=ntl*CYC_HI/FCLK,
               gops_system=ops/tot/1e9, gops_array=ops/t_comp/1e9,
               efficiency=t_comp/tot, verified=ok)
    table.append(row)
    print(f"{size:5d} {ntl:7d} {tot*1e3:11.3f} {t_comp*1e3:10.3f} "
          f"{t_over*1e3:10.3f} {100*t_comp/tot:6.2f}% "
          f"{ops/tot/1e9:9.4f} {ops/t_comp/1e9:9.4f}")

print()
print(f"cosimulation bound on T_compute at the largest size: "
      f"{table[-1]['t_compute_lo_s']*1e3:.3f} to "
      f"{table[-1]['t_compute_hi_s']*1e3:.3f} ms")
print(f"efficiency is flat at {np.mean([r['efficiency'] for r in table])*100:.2f}% "
      f"if the per tile cost never amortises")

import csv
with open("grid_scaling.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(table[0].keys()))
    w.writeheader()
    w.writerows(table)
print("\nwritten to grid_scaling.csv")


 size   tiles  T_total ms  T_comp ms  T_over ms  comp %  GOPS sys  GOPS arr
---------------------------------------------------------------------------
    4       1       0.020      0.000      0.019   1.28%    0.0065    0.5120
    8       8       0.112      0.002      0.110   1.78%    0.0091    0.5120
   16      64       0.894      0.016      0.878   1.79%    0.0092    0.5120
   32     512       8.160      0.128      8.032   1.57%    0.0080    0.5120
   64    4096      59.429      1.024     58.405   1.72%    0.0088    0.5120
  128   32768     477.106      8.192    468.914   1.72%    0.0088    0.5120

cosimulation bound on T_compute at the largest size: 7.318 to 17.040 ms
efficiency is flat at 1.64% if the per tile cost never amortises

written to grid_scaling.csv


In [9]:
if rc is not None:
    rc.shutdown()
    await cluster.stop_cluster()
await cluster.clean_cluster()
await cluster.release_cluster()
print("released")

Stopping controller
Controller stopped: {'exit_code': 0, 'pid': 9330, 'identifier': 'ipcontroller-1787084275-86qw-106'}
Stopping engine(s): 1787084276
released
